In [1]:
from pathlib import Path
from typing import List, Tuple, Dict
import json, random, itertools, statistics as st
import seqeval

import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import KFold, train_test_split
from collections import Counter
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import BallTree
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity, pairwise_distances
from sklearn.cluster import KMeans

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)

from seqeval.metrics import f1_score, classification_report
from evaluate import load as load_metric
from tqdm.auto import tqdm

/home/hartb/Estudos/mestrado/ner-splits/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Configuração e Verificação Inicial

In [2]:
SEED_GLOBAL = 42
random.seed(SEED_GLOBAL)
np.random.seed(SEED_GLOBAL)
torch.manual_seed(SEED_GLOBAL)

CSV_PATH = Path("data/cachacaNER.csv")  # ajuste se necessário
MODEL_NAME = "neuralmind/bert-base-portuguese-cased"

df = pd.read_csv(CSV_PATH)

In [3]:
for cand in ["sentence_id", "sentence", "sent_id"]:
    if cand in df.columns:
        SENT_COL = cand
        break
else:
    raise KeyError(f"Coluna de sentença não encontrada em {list(df.columns)}")


def sent_to_record(sent_id, g):
    return {
        "sentence_id": int(sent_id),
        "tokens": g["token"].tolist(),
        "ner_tags": g["tag"].tolist(),
    }


In [4]:
records = [sent_to_record(i, g) for i, g in df.groupby(SENT_COL, sort=False)]
cachaca_full = Dataset.from_list(records)

In [5]:
cachaca_full

Dataset({
    features: ['sentence_id', 'tokens', 'ner_tags'],
    num_rows: 13628
})

In [6]:
# lista de rótulos (ordem alfabética garante consistência entre runs)
label_list = sorted({lab for sent in cachaca_full["ner_tags"] for lab in sent})
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}
NUM_LABELS = len(label_list)

In [7]:
id2label

{0: 'B-CARACTERISTICA_SENSORIAL_AROMA',
 1: 'B-CARACTERISTICA_SENSORIAL_CONSISTÊNCIA',
 2: 'B-CARACTERISTICA_SENSORIAL_COR',
 3: 'B-CARACTERISTICA_SENSORIAL_SABOR',
 4: 'B-CLASSIFICACAO_BEBIDA',
 5: 'B-EQUIPAMENTO_DESTILACAO',
 6: 'B-GRADUACAO_ALCOOLICA',
 7: 'B-NOME_BEBIDA',
 8: 'B-NOME_LOCAL',
 9: 'B-NOME_ORGANIZACAO',
 10: 'B-NOME_PESSOA',
 11: 'B-PRECO',
 12: 'B-RECIPIENTE_ARMAZENAMENTO',
 13: 'B-TEMPO',
 14: 'B-TEMPO_ARMAZENAMENTO',
 15: 'B-TIPO_MADEIRA',
 16: 'B-VOLUME',
 17: 'I-CARACTERISTICA_SENSORIAL_AROMA',
 18: 'I-CARACTERISTICA_SENSORIAL_CONSISTÊNCIA',
 19: 'I-CARACTERISTICA_SENSORIAL_COR',
 20: 'I-CARACTERISTICA_SENSORIAL_SABOR',
 21: 'I-CLASSIFICACAO_BEBIDA',
 22: 'I-EQUIPAMENTO_DESTILACAO',
 23: 'I-GRADUACAO_ALCOOLICA',
 24: 'I-NOME_BEBIDA',
 25: 'I-NOME_LOCAL',
 26: 'I-NOME_ORGANIZACAO',
 27: 'I-NOME_PESSOA',
 28: 'I-PRECO',
 29: 'I-RECIPIENTE_ARMAZENAMENTO',
 30: 'I-TEMPO',
 31: 'I-TEMPO_ARMAZENAMENTO',
 32: 'I-TIPO_MADEIRA',
 33: 'I-VOLUME',
 34: 'O'}

In [8]:
NUM_LABELS

35

# Splits

In [9]:
# ---------- 1) Heuristic Length  --------------------------------------
def heur_len_split(
    dataset: Dataset, pct_test: float = 0.20, pct_val: float = 0.10, seed: int = 42
) -> DatasetDict:
    """Testa só sentenças ≥ máx(len_train)."""
    sent_lens = np.array([len(t) for t in dataset["tokens"]])
    # separação inicial train/dev (randômica estratificando por tamanho grosso)
    idx_all = np.arange(len(dataset))
    train_idx, temp_idx = train_test_split(
        idx_all,
        test_size=pct_test + pct_val,
        stratify=(sent_lens // 5),  # bin len
        random_state=seed,
    )
    # define longo-threshold como tamanho máx. do treino
    max_train_len = sent_lens[train_idx].max()
    # test = sentenças > threshold.  Caso falte/ sobre exemplos, ajusta.
    test_idx = [i for i in temp_idx if sent_lens[i] > max_train_len]
    resto_idx = [i for i in temp_idx if i not in test_idx]
    # completa ou reduz para atingir pct_test
    need = int(pct_test * len(dataset)) - len(test_idx)
    if need > 0:
        test_idx.extend(resto_idx[:need])
        val_idx = resto_idx[need:]
    else:
        val_keep = int(pct_val * len(dataset))
        val_idx = resto_idx[:val_keep]
        test_idx = test_idx[: int(pct_test * len(dataset))]
    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )


# ---------- 2) Heuristic Rare-Words ----------------------------------
def heur_rare_split(
    dataset: Dataset, pct_test: float = 0.20, pct_val: float = 0.10, seed: int = 42
) -> DatasetDict:
    """Testa frases que contenham palavras do quintil + raro."""
    # contagem de frequência de token
    freqs = Counter(w for toks in dataset["tokens"] for w in toks)
    # define rareza: 20 % mais raras
    thresh = np.quantile(list(freqs.values()), 0.20)
    rare_set = {w for w, c in freqs.items() if c <= thresh}
    is_rare = np.array([any(w in rare_set for w in toks) for toks in dataset["tokens"]])
    rare_idx = np.where(is_rare)[0]
    common_idx = np.where(~is_rare)[0]
    # garante proporções desejadas
    n_test = int(pct_test * len(dataset))
    n_val = int(pct_val * len(dataset))
    rng = np.random.default_rng(seed)
    test_idx = rng.choice(rare_idx, size=min(len(rare_idx), n_test), replace=False)
    resto_idx = [i for i in rare_idx if i not in test_idx] + list(common_idx)
    val_idx = rng.choice(resto_idx, size=n_val, replace=False)
    train_idx = [i for i in resto_idx if i not in val_idx]
    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )


# ---------- 3) Standard (80-10-10) -----------------------------------
def std_split(
    dataset: Dataset, pct_test: float = 0.10, pct_val: float = 0.10, seed: int = 42
) -> DatasetDict:
    """Split aleatório estratificado por comprimento (PTB-like)."""
    idx = np.arange(len(dataset))
    strat = np.array([len(t) for t in dataset["tokens"]]) // 5
    train_idx, temp_idx = train_test_split(
        idx, test_size=pct_test + pct_val, stratify=strat, random_state=seed
    )
    val_rel = pct_val / (pct_test + pct_val)
    val_idx, test_idx = train_test_split(
        temp_idx, test_size=1 - val_rel, stratify=strat[temp_idx], random_state=seed
    )
    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )


# ---------- 4) Adversarial (approx. Wasserstein) ---------------------
def adversarial_split(
    dataset: Dataset,
    pct_test: float = 0.20,
    pct_val: float = 0.10,
    k: int = 5,
    seed: int = 42,
) -> DatasetDict:
    """
    Seleciona k frases 'mais distantes' recursivamente (BallTree + W₂)
    para compor o teste, lembrando Alg.-1 de Søgaard et al.【turn6file4】.
    """
    # SBERT embed (rápido na GPU / aceitável CPU)
    sbert = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    emb = sbert.encode(
        [" ".join(toks) for toks in dataset["tokens"]],
        batch_size=64,
        show_progress_bar=False,
    )
    # BallTree para vizinhança eficiente
    tree = BallTree(emb, leaf_size=40)
    idx_pool = set(range(len(dataset)))
    test_idx = []
    rng = np.random.default_rng(seed)
    while len(test_idx) < int(pct_test * len(dataset)):
        # amostra candidata: ponto + longe do centro
        center = emb[list(idx_pool)].mean(0, keepdims=True)
        dists, _ = tree.query(center, k=len(idx_pool))
        farthest = list(idx_pool)[int(dists.argmax())]
        # pega-se farest e seus k-NN mais próximos  ⇒ aumenta diversidade
        nn = tree.query([emb[farthest]], k=k, return_distance=False)[0]
        for j in nn:
            if j in idx_pool and len(test_idx) < int(pct_test * len(dataset)):
                test_idx.append(j)
                idx_pool.remove(j)
    # retira val
    val_size = int(pct_val * len(dataset))
    val_idx = rng.choice(list(idx_pool), size=val_size, replace=False)
    idx_pool -= set(val_idx)
    train_idx = list(idx_pool)
    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [10]:
def heur_len_split(
    dataset: Dataset,
    pct_test: float = 0.20,
    pct_val: float = 0.10,
    seed: int = 42,
    bin_size: int = 10,
) -> DatasetDict:
    """
    Teste = sentenças mais longas que max(len(train)).
    Robusto a datasets pequenos: se estratificação falhar, usa split aleatório.
    """
    rng = np.random.default_rng(seed)
    idx_all = np.arange(len(dataset))
    sent_lens = np.array([len(t) for t in dataset["tokens"]])

    # ------ 1) tenta split estratificado por baldes -------------------------
    strat = sent_lens // bin_size
    try:
        train_idx, temp_idx = train_test_split(
            idx_all,
            test_size=pct_test + pct_val,
            stratify=strat,
            random_state=seed,
        )
    except ValueError:  # classes com 1 amostra
        train_idx, temp_idx = train_test_split(
            idx_all,
            test_size=pct_test + pct_val,
            shuffle=True,
            random_state=seed,
            stratify=None,
        )

    # ------ 2) escolhe test = len > max(train) ------------------------------
    max_train_len = sent_lens[train_idx].max()
    test_idx = [i for i in temp_idx if sent_lens[i] > max_train_len]
    resto_idx = [i for i in temp_idx if i not in test_idx]

    # garante tamanhos exatos
    n_test_desired = int(pct_test * len(dataset))
    n_val_desired = int(pct_val * len(dataset))

    # completa teste se ficou pequeno
    if len(test_idx) < n_test_desired:
        extra = rng.choice(
            resto_idx, size=n_test_desired - len(test_idx), replace=False
        )
        test_idx.extend(extra)
        resto_idx = [i for i in resto_idx if i not in extra]

    # define validação
    val_idx = rng.choice(resto_idx, size=n_val_desired, replace=False)
    train_idx = [i for i in idx_all if i not in test_idx and i not in val_idx]

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [11]:
def loc_split(
    dataset: Dataset,
    pct_test: float = 0.20,
    pct_val: float = 0.10,
    ngram: int = 4,
    seed: int = 42,
) -> DatasetDict:
    """
    Divide por sobreposição léxica (n-gram Jaccard).
    Frações independentes para teste e validação.
    """
    docs = [" ".join(toks) for toks in dataset["tokens"]]

    vect = CountVectorizer(
        analyzer="word", ngram_range=(ngram, ngram), binary=True
    ).fit(docs)
    X = vect.transform(docs)
    bin_counts = X.sum(axis=1).A1

    sim = cosine_similarity(X, dense_output=False)
    k = 5
    topk = np.zeros(len(dataset))
    for i in range(sim.shape[0]):
        row = sim.getrow(i).toarray()[0]
        idx = np.argpartition(-row, range(1, k + 1))[1 : k + 1]
        inter = row[idx] * bin_counts[i]
        uni = bin_counts[i] + bin_counts[idx] - inter
        topk[i] = (inter / uni).mean()

    order = np.argsort(topk)  # baixo → alto overlap
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    test_idx = order[:n_test]
    val_idx = order[n_test : n_test + n_val]
    train_idx = order[n_test + n_val :]

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [12]:
def semantic_cluster_split(
    dataset: Dataset,
    pct_test: float = 0.20,
    pct_val: float = 0.10,
    k: int | None = None,
    seed: int = 42,
) -> DatasetDict:
    """
    Clusters SBERT → reserva clusters distantes para test/val.
    """
    if k is None:
        k = int(np.sqrt(len(dataset)))

    sbert = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    embeddings = sbert.encode(
        [" ".join(t) for t in tqdm(dataset["tokens"])],
        batch_size=64,
        show_progress_bar=False,
    )

    km = KMeans(n_clusters=k, random_state=seed, n_init=10).fit(embeddings)
    labels = km.labels_
    centroids = km.cluster_centers_

    global_center = embeddings.mean(0, keepdims=True)
    dists = pairwise_distances(centroids, global_center).flatten()

    clusters_sorted = np.argsort(-dists)  # mais distantes primeiro
    test_clusters, val_clusters = set(), set()
    total_test = total_val = 0
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    for c in clusters_sorted:
        size = np.sum(labels == c)
        if total_test < n_test:  # preenche primeiro o teste
            test_clusters.add(c)
            total_test += size
        elif total_val < n_val:  # depois a validação
            val_clusters.add(c)
            total_val += size
        if total_test >= n_test and total_val >= n_val:
            break

    test_idx = np.where([lbl in test_clusters for lbl in labels])[0]
    val_idx = np.where([lbl in val_clusters for lbl in labels])[0]
    train_idx = np.where(
        [lbl not in test_clusters and lbl not in val_clusters for lbl in labels]
    )[0]

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [13]:
def difficulty_scores(dataset: Dataset) -> np.ndarray:
    length = np.array([len(tok) for tok in dataset["tokens"]], dtype=float)
    length = (length - length.mean()) / length.std()

    dens = []
    for labels in dataset["ner_tokens"]:
        non_o = sum(1 for l in labels if l != "O")
        dens.append(non_o / len(labels))
    dens = np.array(dens)
    dens = (dens - dens.mean()) / dens.std()

    ent_types, freq = [], Counter()
    for labels in dataset["ner_tokens"]:
        types = [l[2:] for l in labels if l != "O"]
        ent_types.append(types[0] if types else "NONE")
    freq.update(ent_types)
    rarity = np.array([1 / freq[t] for t in ent_types])
    rarity = (rarity - rarity.mean()) / rarity.std()

    return length + dens + rarity


def reverse_curriculum_split(
    dataset: Dataset, pct_test: float = 0.20, pct_val: float = 0.10, seed: int = 42
) -> DatasetDict:
    scores = difficulty_scores(dataset)
    order = np.argsort(scores)  # easy→hard
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    test_idx = order[-n_test:]  # hardest
    val_idx = order[-(n_test + n_val) : -n_test]
    train_idx = order[: -(n_test + n_val)]

    rng = np.random.RandomState(seed)
    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    rng.shuffle(test_idx)

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [14]:
# standard_split = split_standard(cachaca_full)
# print('std')
# random_splt = random_splits(cachaca_full)
# print('random')
heur_len = heur_len_split(cachaca_full)
print("heur_len")
# heur_rare = split_heur_rare(cachaca_full)
# print("heur_rare")
# advers = split_adversarial_fast(cachaca_full)
# print("advs")
# print("loc")
# loc = loc_split(cachaca_full)

heur_len


# Experimentos

In [15]:
from sklearn.metrics import f1_score as skl_f1

In [16]:
def train_ner_with_split(
    dataset: Dataset,
    split: str,  # "loc" | "semantic" | "reverse" | func
    pct_test: float = 0.20,
    pct_val: float = 0.10,
    model_ckpt: str = "Davlan/distilbert-base-multilingual-cased-ner-hrl",
    training_args_kwargs: dict | None = None,
    split_kwargs: dict | None = None,
    seed: int = 42,
):
    """
    Treina um modelo de NER usando a estratégia de split desejada.
    Retorna (trainer, métricas_test).
    """
    # 1. Escolhe função de split ------------------------------------------------
    if callable(split):
        split_fn = split
    else:
        _map = {
            "loc": loc_split,
            "reverse": reverse_curriculum_split,
            "semantic": semantic_cluster_split,
            "heur_len": heur_len_split,
            "heur_rare": heur_rare_split,
            "std": std_split,
            "advs": adversarial_split,
        }
        if split not in _map:
            raise ValueError(f"split='{split}' não reconhecido.")
        split_fn = _map[split]

    split_kwargs = split_kwargs or {}
    ds = split_fn(
        dataset, pct_test=pct_test, pct_val=pct_val, seed=seed, **split_kwargs
    )  # train/val/test

    # 2. Tokenizer e modelo -----------------------------------------------------
    label_list = sorted({l for labels in dataset["ner_tags"] for l in labels})
    label2id = {l: i for i, l in enumerate(label_list)}
    id2label = {i: l for l, i in label2id.items()}

    num_labels = len(label_list)
    tok = AutoTokenizer.from_pretrained(model_ckpt)
    model = AutoModelForTokenClassification.from_pretrained(
        model_ckpt,
        num_labels=num_labels,  # ← adapta o tamanho
        ignore_mismatched_sizes=True,  # ← descarta pesos velhos da head
    )
    model.config.id2label = id2label
    model.config.label2id = label2id

    # 3. Mapeamento label↔id ----------------------------------------------------
    # label_list = sorted(
    #     {l for labels in dataset["ner_tags"] for l in labels if l != "O"}
    # )

    # 4. Tokenização + alinhamento ---------------------------------------------
    # def tok_function(ex):
    #     return tok(
    #         ex["tokens"], is_split_into_words=True, truncation=True, padding=False
    #     )

    # def align_labels(ex):
    #     word_ids = ex.word_ids()
    #     labels = []
    #     for w in word_ids:
    #         if w is None:
    #             labels.append(-100)
    #         else:
    #             labels.append(label2id.get(ex["ner_tags"][w], 0))
    #     ex["labels"] = labels
    #     return ex

    # ds_tok = ds.map(tok_function, batched=True)
    # ds_tok = ds_tok.map(align_labels)

    def tokenize_and_align_labels(examples, label_all_tokens=False):
        tokenized = tok(examples["tokens"], is_split_into_words=True, truncation=True)

        labels_batch = []
        for i, word_labels in enumerate(examples["ner_tags"]):
            word_ids = tokenized.word_ids(batch_index=i)  # <- aqui sim
            label_ids = []
            previous_word_idx = None
            for word_idx in word_ids:
                if word_idx is None:
                    label_ids.append(-100)  # máscara
                elif word_idx != previous_word_idx:
                    label_ids.append(label2id[word_labels[word_idx]])
                else:
                    # marca sub-tokens; mude para `label2id[...]`
                    # se quiser repetir label em todos os sub-tokens
                    label_ids.append(
                        label2id[word_labels[word_idx]] if label_all_tokens else -100
                    )
                previous_word_idx = word_idx
            labels_batch.append(label_ids)

        tokenized["labels"] = labels_batch
        return tokenized

    ds_tok = ds.map(
        tokenize_and_align_labels, batched=True, remove_columns=ds["train"].column_names
    )
    # 5. Métrica (seqeval) ------------------------------------------------------
    seqeval = load_metric("seqeval")

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)

        sent_preds, sent_labels = [], []  # p/ seqeval
        flat_preds, flat_labels = [], []  # p/ sklearn

        for p_row, l_row in zip(preds, labels):
            p_sent, l_sent = [], []
            for pi, li in zip(p_row, l_row):
                if li != -100:
                    lbl_true = id2label[li]
                    lbl_pred = id2label[pi]
                    p_sent.append(lbl_pred)
                    l_sent.append(lbl_true)
                    flat_preds.append(lbl_pred)
                    flat_labels.append(lbl_true)
            sent_preds.append(p_sent)
            sent_labels.append(l_sent)

        # métricas seqeval (micro F1 = overall_f1)
        seqeval_metrics = seqeval.compute(
            predictions=sent_preds,
            references=sent_labels,
        )

        # métricas sklearn
        f1_micro = skl_f1(flat_labels, flat_preds, average="micro", zero_division=0)
        f1_macro = skl_f1(flat_labels, flat_preds, average="macro", zero_division=0)
        f1_weighted = skl_f1(
            flat_labels, flat_preds, average="weighted", zero_division=0
        )

        return {
            **seqeval_metrics,  # overall_precision / recall / f1
            "f1_micro": f1_micro,
            "f1_macro": f1_macro,
            "f1_weighted": f1_weighted,
        }

    # 6. Args de treinamento ----------------------------------------------------
    args_defaults = dict(
        output_dir=f"ner-{split}",
        # estratégia de avaliação + salvamento
        eval_strategy="epoch",  # novo nome (4.52+)
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="overall_f1",
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=5,
        seed=seed,
        report_to="none"
    )

    if training_args_kwargs:
        args_defaults.update(training_args_kwargs)
    args = TrainingArguments(**args_defaults)

    data_collator = DataCollatorForTokenClassification(tokenizer=tok, padding=True)

    # 7. Trainer ---------------------------------------------------------------
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=ds_tok["train"],
        eval_dataset=ds_tok["val"],
        tokenizer=tok,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    # 8. Avaliação final --------------------------------------------------------
    test_metrics = trainer.evaluate(eval_dataset=ds_tok["test"])
    return trainer, test_metrics

In [17]:
trainer, loc_metrics = train_ner_with_split(
    cachaca_full, split="heur_len"#, split_kwargs={"ngram": 4}
)

Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at Davlan/distilbert-base-multilingual-cased-ner-hrl and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([9]) in the checkpoint and torch.Size([35]) in the model instantiated
- classifier.weight: found shape torch.Size([9, 768]) in the checkpoint and torch.Size([35, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 2725/2725 [00:00<00:00, 18339.40 examples/s]
/tmp/ipykernel_126060/1315278726.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


OutOfMemoryError: CUDA out of memory. Tried to allocate 352.00 MiB. GPU 0 has a total capacity of 3.81 GiB of which 310.69 MiB is free. Process 124937 has 3.46 GiB memory in use. Including non-PyTorch memory, this process has 50.00 MiB memory in use. Of the allocated memory 0 bytes is allocated by PyTorch, and 0 bytes is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
loc_metrics

{'eval_loss': 0.20134669542312622,
 'eval_CARACTERISTICA_SENSORIAL_AROMA': {'precision': 0.635593220338983,
  'recall': 0.646551724137931,
  'f1': 0.641025641025641,
  'number': 464},
 'eval_CARACTERISTICA_SENSORIAL_CONSISTÊNCIA': {'precision': 0.8080808080808081,
  'recall': 0.7339449541284404,
  'f1': 0.7692307692307693,
  'number': 109},
 'eval_CARACTERISTICA_SENSORIAL_COR': {'precision': 0.7692307692307693,
  'recall': 0.7575757575757576,
  'f1': 0.7633587786259541,
  'number': 198},
 'eval_CARACTERISTICA_SENSORIAL_SABOR': {'precision': 0.5850746268656717,
  'recall': 0.4336283185840708,
  'f1': 0.49809402795425667,
  'number': 452},
 'eval_CLASSIFICACAO_BEBIDA': {'precision': 0.7985347985347986,
  'recall': 0.8790322580645161,
  'f1': 0.8368522072936659,
  'number': 248},
 'eval_EQUIPAMENTO_DESTILACAO': {'precision': 0.746268656716418,
  'recall': 0.7692307692307693,
  'f1': 0.7575757575757576,
  'number': 65},
 'eval_GRADUACAO_ALCOOLICA': {'precision': 0.8596491228070176,
  'reca

In [ ]:
loc_metrics.keys()

dict_keys(['eval_loss', 'eval_CARACTERISTICA_SENSORIAL_AROMA', 'eval_CARACTERISTICA_SENSORIAL_CONSISTÊNCIA', 'eval_CARACTERISTICA_SENSORIAL_COR', 'eval_CARACTERISTICA_SENSORIAL_SABOR', 'eval_CLASSIFICACAO_BEBIDA', 'eval_EQUIPAMENTO_DESTILACAO', 'eval_GRADUACAO_ALCOOLICA', 'eval_NOME_BEBIDA', 'eval_NOME_LOCAL', 'eval_NOME_ORGANIZACAO', 'eval_NOME_PESSOA', 'eval_PRECO', 'eval_RECIPIENTE_ARMAZENAMENTO', 'eval_TEMPO', 'eval_TEMPO_ARMAZENAMENTO', 'eval_TIPO_MADEIRA', 'eval_VOLUME', 'eval_overall_precision', 'eval_overall_recall', 'eval_overall_f1', 'eval_overall_accuracy', 'eval_f1_micro', 'eval_f1_macro', 'eval_f1_weighted', 'eval_runtime', 'eval_samples_per_second', 'eval_steps_per_second', 'epoch'])

In [ ]:
print("F1 Macro:", loc_metrics["eval_f1_micro"])

F1 Macro: 0.9579240496671689


In [ ]:
print("F1 Micro:", loc_metrics["eval_f1_macro"])

F1 Micro: 0.7701053003002256


In [ ]:
print("F1 Weight:", loc_metrics["eval_f1_weighted"])

F1 Weight: 0.9559846011861013
